# 02. Map-Reduce 요약

긴 문서를 여러 조각(chunk)으로 나눈 뒤 두 단계로 요약합니다.

1. **Map**: 각 chunk 를 **병렬로** 개별 요약(여기서는 핵심 내용 추출)합니다.
2. **Reduce**: Map 결과들을 모아 **하나의 최종 요약**으로 합칩니다.

LLM 의 컨텍스트 창보다 긴 문서도 요약할 수 있고, Map 단계를 병렬 처리(`batch`)하므로 빠릅니다.

## 환경 설정

- API 키(`OPENAI_API_KEY`, `LANGSMITH_API_KEY` 등)는 `.env` 파일 또는 시스템 환경변수에서 읽어옵니다.
- `logging.langsmith()` 를 호출하면 실행 과정이 LangSmith 에 기록되어 체인 내부 동작을 확인할 수 있습니다.

In [1]:
# .env 파일에 저장된 API KEY 를 환경변수로 불러옵니다.
from dotenv import load_dotenv

load_dotenv()

# LangSmith 추적 설정 (프로젝트 이름: Summary)
from langchain_teddynote import logging

logging.langsmith("Summary")

LangSmith 추적을 시작합니다.
[프로젝트명]
Summary


## 데이터 로드

**실습 문서**: 소프트웨어정책연구소(SPRi) AI Brief 2023년 12월호 (`data/SPRI_AI_Brief_2023년12월호_F.pdf`)

In [2]:
from langchain_community.document_loaders import PyPDFLoader

# PDF 를 페이지 단위로 로드합니다. (1페이지 = Document 1개)
loader = PyPDFLoader("../data/SPRI_AI_Brief_2023년12월호_F.pdf")
docs = loader.load()
print(f"전체 페이지 수: {len(docs)}")

# 실습 비용/시간을 줄이기 위해 일부 페이지(4~8쪽)만 요약합니다.
docs = docs[3:8]
print(f"요약할 페이지 수: {len(docs)}")

C:\Users\user\AppData\Local\Temp\ipykernel_8560\1564795920.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


전체 페이지 수: 23
요약할 페이지 수: 5


### 프롬프트 가져오기 (LangSmith Hub)

> ⚠️ **레거시 변경점**
> 예전 코드의 `from langchain import hub` / `hub.pull(...)` 은 LangChain 1.x 에서 `langchain_classic` 으로 밀려났고,
> 최신 `langsmith` 는 **다른 사람(owner)의 공개 프롬프트**를 기본적으로 막습니다.
> 그래서 `langsmith.Client().pull_prompt(..., dangerously_pull_public_prompt=True)` 로 직접 가져옵니다.
> (`teddynote` 튜토리얼 프롬프트처럼 **신뢰할 수 있는 프롬프트에만** 사용하세요.)

In [3]:
from langsmith import Client

client = Client()


def pull_prompt(name: str):
    """LangSmith Hub 에서 공개 프롬프트를 가져옵니다. (hub.pull 대체)"""
    return client.pull_prompt(name, dangerously_pull_public_prompt=True)

## Map 단계

각 페이지에서 핵심 주장(thesis)을 1~5개 추출합니다.
(정석은 "요약"이지만 어차피 Reduce 에서 다시 합치므로 "핵심 내용 추출"로 해도 됩니다.)

In [4]:
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# map 프롬프트 (입력 변수: doc)
map_prompt = pull_prompt("teddynote/map-prompt")
map_prompt.pretty_print()

================================ System Message ================================

You are a professional main thesis extractor.

================================ Human Message =================================

Your task is to extract main thesis from given documents. Answer should be in same language as given document. 

#Format: 
- thesis 1
- thesis 2
- thesis 3
- ...

Here is a given document: 
{doc}

Write 1~5 sentences.
#Answer:


In [5]:
# map 체인: 프롬프트 → LLM → 문자열
map_chain = map_prompt | llm | StrOutputParser()

`batch()` 에 입력 리스트를 넘기면 **여러 문서를 병렬로** 처리합니다.

> 💡 Document 객체를 그대로 넘기면 `metadata` 까지 문자열로 섞여 들어가므로, `page_content` 만 넘깁니다.

In [6]:
# 각 페이지의 핵심 내용 추출 (병렬 실행)
doc_summaries = map_chain.batch([{"doc": doc.page_content} for doc in docs])

print(f"map 결과 수: {len(doc_summaries)}\n")
print(doc_summaries[0])  # 첫 번째 페이지 결과

map 결과 수: 5

- 미국 바이든 대통령이 안전하고 신뢰할 수 있는 AI 개발과 사용을 보장하기 위한 행정명령을 발표하였다.
- 이 행정명령은 AI의 안전과 보안 기준, 개인정보 보호, 형평성과 시민권 향상, 소비자 보호, 노동자 지원, 혁신과 경쟁 촉진, 국제 협력을 포함한 광범위한 내용을 담고 있다.
- AI 시스템 개발 기업은 안전 테스트 결과와 시스템 정보를 미국 정부와 공유해야 하며, AI의 책임 있는 사용을 위한 법적 지침이 마련될 예정이다.
- 또한, AI 연구를 촉진하고 중소기업과 개발자에게 기술과 인프라를 지원하기 위한 국가 AI 연구자원 프로그램이 추진된다.


## Reduce 단계

Map 단계에서 뽑은 핵심 내용들을 하나의 최종 요약으로 통합합니다.

In [7]:
# reduce 프롬프트 (입력 변수: doc_summaries, language)
reduce_prompt = pull_prompt("teddynote/reduce-prompt")
reduce_prompt.pretty_print()

================================ System Message ================================

You are a professional summarizer. You are given a list of summaries of documents and you are asked to create a single summary of the documents.

================================ Human Message =================================

#Instructions: 
1. Extract main points from a list of summaries of documents
2. Make final summaries in bullet points format.
3. Answer should be written in {language}.

#Format: 
- summary 1
- summary 2
- summary 3
- ...

Here is a list of summaries of documents: 
{doc_summaries}

#SUMMARY:


In [8]:
from langchain_teddynote.messages import stream_response

reduce_chain = reduce_prompt | llm | StrOutputParser()

# map 결과들을 줄바꿈으로 이어 붙여 한 번에 전달하고, 결과를 스트리밍 출력합니다.
answer = stream_response(
    reduce_chain.stream({"doc_summaries": "\n".join(doc_summaries), "language": "Korean"}),
    return_output=True,
)

- 미국 바이든 대통령이 AI 개발과 사용의 안전성을 보장하기 위한 행정명령을 발표하였다.
- 행정명령은 AI의 안전 기준, 개인정보 보호, 형평성, 소비자 보호, 노동자 지원, 혁신 촉진, 국제 협력을 포함한다.
- AI 시스템 개발 기업은 안전 테스트 결과를 정부와 공유해야 하며, 법적 지침이 마련될 예정이다.
- 국가 AI 연구자원 프로그램이 추진되어 중소기업과 개발자에게 지원이 이루어진다.
- G7은 '히로시마 AI 프로세스'를 통해 AI 기업을 위한 국제 행동강령에 합의하였다.
- 행동강령은 AI의 위험 평가, 투명성 보장, 정보 공유를 요구한다.
- G7은 첨단 AI 시스템의 자발적 채택을 권장하고, 이해관계자 협의를 통해 행동강령을 개정할 예정이다.
- 28개국이 AI 안전성 정상회의에서 AI 안전 보장을 위한 블레츨리 선언을 발표하였다.
- 선언은 AI 시스템의 안전을 위해 모든 이해관계자의 협력이 중요하다고 강조하였다.
- 영국 총리는 AI 안전 연구소 출범과 정부 주도의 안전 테스트 계획을 발표하였다.
- 각국은 AI 안전 보장을 위해 협력하기로 합의하였다.
- 한국은 영국과 AI 미니 정상회의를 공동 개최하고, 프랑스와 대면 정상회의를 개최할 예정이다.
- 미국 법원은 생성 AI 기업에 대한 저작권 침해 소송을 기각하였으나, 일부 소송은 계속 진행된다.
- FTC는 생성 AI로 인한 소비자 피해와 빅테크의 시장 지배력 우려를 표명하였다.
- FTC는 AI 관련 불법 행위에 대응할 것임을 강조하였다.

## Map-Reduce 를 하나의 체인으로 묶기

`@chain` 데코레이터를 붙이면 일반 파이썬 함수를 `Runnable` 로 만들 수 있습니다.
→ `.invoke()` 로 호출할 수 있고, LangSmith 에도 하나의 실행 단위로 기록됩니다.

In [9]:
from langchain_core.runnables import chain


@chain
def map_reduce_chain(docs):
    """Document 리스트를 받아 Map-Reduce 방식으로 요약합니다."""
    # 1) Map: 각 문서의 핵심 내용을 병렬로 추출
    map_chain = pull_prompt("teddynote/map-prompt") | llm | StrOutputParser()
    doc_summaries = map_chain.batch([{"doc": doc.page_content} for doc in docs])

    # 2) Reduce: 추출한 내용을 하나의 요약으로 통합
    #    (최종 요약 품질을 높이고 싶다면 여기만 gpt-4o 같은 상위 모델로 바꿔도 됩니다.)
    reduce_chain = pull_prompt("teddynote/reduce-prompt") | llm | StrOutputParser()
    return reduce_chain.invoke(
        {"doc_summaries": "\n".join(doc_summaries), "language": "Korean"}
    )

In [10]:
answer = map_reduce_chain.invoke(docs)
print(answer)

- 미국 바이든 대통령이 AI 개발과 사용을 위한 행정명령을 발표, 안전과 신뢰성 보장.
- 행정명령은 AI의 안전 기준, 개인정보 보호, 형평성, 소비자 보호, 노동자 지원, 혁신 촉진, 국제 협력을 포함.
- AI 시스템 개발 기업은 안전 테스트 결과를 정부와 공유해야 하며, 법적 지침이 마련됨.
- 의료 분야에서 AI 사용을 촉진하고 근로자 피해 완화 원칙 설정.
- 국가AI연구자원(NAIRR)을 통해 AI 연구 촉진 및 외국인 전문가 지원 방안 포함.
- G7은 '히로시마 AI 프로세스'를 통해 AI 기업을 위한 국제 행동강령에 합의.
- 행동강령은 AI 수명주기 전반의 위험 평가, 투명성 보장, 정보공유 요구.
- G7은 첨단 AI 시스템 개발에 대한 자발적 채택 권장 및 행동강령 개정 예정.
- 28개국이 AI 안전성 정상회의에서 AI 안전 보장 협력 방안 담은 블레츨리 선언 발표.
- 선언은 AI 시스템 안전 보장을 위한 이해관계자 협력의 중요성 강조.
- 영국 총리는 AI 안전 연구소 출범 및 정부 주도의 안전 테스트 계획 수립 발표.
- 각국은 AI 안전 보장을 위해 투명성 향상, 평가지표 개발, 공공부문 역량 구축 협력 합의.
- 한국은 영국과 AI 미니 정상회의, 프랑스와 대면 정상회의 개최 예정.
- 미국 법원은 생성 AI 기업에 대한 저작권 침해 소송 기각, 저작권 등록 미비 및 유사성 입증 어려움 지적.
- 판사는 원고 측에 고소장 수정 요청.
- 미국 FTC는 생성 AI로 인한 소비자 및 창작자 피해 가능성 및 빅테크 시장 지배력 우려 표명.
- FTC는 생성 AI의 위험 경고 및 법적 권한을 통한 불법 행위 대응 강조.
